Token Embeddings in the simplest form 

it represents the individual token in the dimensions you provide so it acts like a look up table of ,in how many ways the token can be identified

the parameters are the 

vocab_size -> this represents how many rows (as the total size of the dictionary where the model can use this generating the words)

num_dims -> this is the number of dimensions in which the individual word is represented 

we need to explicitly return the params ,or else optimizer can't be able to track any parameters to change in the backprop

actually custom backward is not necessary if you inherited the nn.Module

if you wanted to take a look about how do they change and you can give a try for them

out = weight[indices]

output CHANGES based on WHICH indices are used

When we differentiate, we're looking at the RELATIONSHIP between output and weights

The indices act as a "selector" - they determine WHICH weights affect the output

If you change indices -> you get DIFFERENT rows from weights (discrete change - can't differentiate!)

If you change weights -> the values in those rows change (continuous change - CAN differentiate!)

actually what is the learnable one?weights right not the indices 

so we do use the backprop on the weights not the indices this is the simple way to understand

loss/weights = loss/out * out/weights (we are using the chain rule here)

loss/out = grad_out

self.weights.grad -> this is what needed to be change right and we can change this by the

0        _ _ _ _ .......... _

1        _ _ _ _ .......... _

2        _ _ _ _ .......... _

3        _ _ _ _ .......... _

in this way the weights actually are

and when we bring back the losses and we will bring back with the indices right

so that we can learn to see which indices have actually needed to learn

we got back the two indices 0,2 only and we need to use the backprop on them only

we use the loop for that indices to be targeted ,like looping on them

because we get the list of the indices and with that indices we need to use the backprop individually right

there is a chance of the

accumulating so that we need to add them

this works for the single batch,just start with the single batch

self.weights.grad[idx]+=grad_out[i]

this works in the way we can say that

i -> indices index

idx ->  that specific row

so we need to know the grad_out[index] 

how much does it need to change

and we need to accumulate to that same row

for i,row_idx in enumerate(self.indices):

            self.weights.grad[row_idx]+=grad_out[i]

i->index this is what 0,1,2,3,4 

row_idx is what we go through the indices [0,2,3,5,0] so as the row_idx is 0 two times now the grad[row_idx] accumulates the gardient at the index 

for _ in indices (it will iterate over the values)

we accumulate to the same row,if there same row comes again ,and if not it will start adding fresh

if you wanted to use the torch code then

def backward(self, grad_out):

    # Initialize gradient tensor if None

    if self.weights.grad is None:

        self.weights.grad = torch.zeros_like(self.weights)

    # index_add_ - the PyTorch way!

    self.weights.grad.index_add_(0, self.indices, grad_out)

shape of the indices -> (B,seq_len)

shape of the positions -> (B,seq_len)

after multiplied with the weights 

they change to the shapes of the (B,Seq_len,Dims)

for backward:

y-> out shape -> (B,T,D)

weights shape -> (vocab_size,D)

indices -> (B,seq_len)

dL/dW = gradient with respect to the embedding table

Shape must be: [V, D] (same as W)

dL/dW[i, j] = dL/dY[i] * dY[i]/dW[i, j]

for the gradients

if that specific index exists and we can use the backprop

dy/dw = dL/dy 

or 0 ,if that index didnt existed

In the forward pass, we take the indices as input. Their shape is (batch_size, seq_len), where seq_len is 24 in this case. We then directly form a matrix to print out -- but that doesn't really do much

it's just to visualize what's going on. Because our indices cover every position we need, we display them all.

Now, the backward pass is the next step.

In the backward pass, the indices are going to affect the weight updates. We're using the self.indices that we saved during the forward pass. This time, we need to update all of them.

Let's take an example from the perspective of tokens.

Consider this sentence:

"There is a man sitting on the couch, in which there is a big spring which is set ready for the prank."

When we convert these words into token indices, we need a tokenizer. So let's assume we have one.

The same words repeat, and we need to map them back. The sequence length here is 22 (ignoring spaces, as I've counted). So the input shape is (1, 22) -- one batch of 22 tokens.

The weight matrix, for example, has size 128 (embedding dimension), so the embedding layer is (vocab_size, 128).

Now, for the backward pass:

The same index may appear multiple times in the sequence. For example, the word "the" appears twice.

When that happens, the gradients from both occurrences must be added together because they correspond to the same embedding vector.

So, the update for each index is the sum of all gradients that came from every position where that index appeared.

Also, note that many indices from the vocabulary don't appear in this sentence at all. Their gradients will be zero. 

But we don't need to explicitly assign zeros to them -- we only care about and update the indices that are actually present in the current batch.

indices = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 0, 1, 2, 10, 11, 9, 1, 12, 13, 14, 6, 15]

 Shape: [1, 22]  (batch_size=1, seq_len=22)

Let's say vocab_size = 1000 (our tokenizer has 1000 possible words).

Y[0,0,:] = W[0:]

batch zero,row 0 (which is word the),all columns (we get all the embeddings for that word the)

Y[0,10,:] = W[0:]

this is again the same word the,we get the same embeddings

grad_out = dL/dY  # Shape: [1, 22, 128]

for each row i in W,we are gonna add them all up where the indices == i

grad_W[0,:] = grad_out[0,0,:]+grad_out[0,10,:]

dL/dw = dL/dY * dY/dw

(for the specific row i) we will take the grad_out for that row i,

dL/dw[i,j] = dL/dY[b,s,i] * dY[b,s,j]/dw[i,j]

Y[b, s, :] = W[indices[b, s], :]

Here, s represents the sequence length (seq_len). But does it print for a single index or all indices?

Since s ranges over all positions in the sequence (s -> [0, seq_len-1]), it will print all indices across the entire sequence.

Now for the backward pass, the gradient equation is:

dL/dW[i, j] = dL/dY[b, s, i] * dY[b, s, j] / dW[i, j]

This is fine -- for a specific row i (which corresponds to a particular token index), we compute the gradient contribution.

But we need to do this for all b, s, and i because:

Y has shape (batch_size, seq_len, embedding_dim)

So we loop over all batches (b), all sequence positions (s), and all embedding dimensions (i)

Previously, when we printed Y[b, s, :], we were showing the entire embedding vector for each position.

Now, in the backward pass, we are working with individual dimensions i instead of the full : -- because we need to compute gradients per weight element.

So yes, we change from : to i to handle each dimension separately during backpropagation.

Now, we are going to change only specific columns too, right? Only when there's a mistake or an update needed for them -- isn't that the point?

So for that, we are pinpointing the specific column. Out of the entire 128 dimensions, we are only going to update the dimensions that actually need changes.

The column that has a change will be considered as 1, and all others as 0. So we can take:

dY[b, s, j] / dW[i, j] = 1

because we are only changing that specific position -- it's a direct mapping.

And now we are left with just grad_out (which is dL/dY). So this is how it directly depends only on the grad_out.

So the entire update is essentially just the change in grad_out -- nothing more than that. No other complex factors are involved besides gradient accumulation.

In other words:

For each token index i that appears at position (b, s):

We take the corresponding gradient from grad_out[b, s, :]

We add it to dW[i, :] (the row for that token)

This is just accumulating gradients from all occurrences

So the backward pass for the embedding layer boils down to: scatter-add the gradients from grad_out into the weight matrix rows based on the indices -- and that's it!

Positional Embeddings is very important as we can't truly rely on the token embeddings

because depends on the position of the word,the meaning changes

in a sentence ,a single word can give different meanings depends on its position

subject,object...

Ex:

The dog bit the man

The man bit the dog  (just for example)

In [ ]:
class TokenEmbeddings:
    def __init__(self, vocab_size, num_dims):
        self.weights = torch.randn(vocab_size, num_dims)

        # Using nn.Parameter would register these weights with PyTorch so they appear in model.parameters() and get updated by optimizers.
        # Since this is a raw implementation, I'm using requires_grad=True manually. In the module version, I'll use nn.Parameter

    def forward(self, indices):
        self.indices = indices
        return self.weights[indices]

    def backward(self, grad_out):
        
        self.grad_weights = torch.zeros_like(self.weights)
        
        for i, token_id in enumerate(self.indices.flatten()):
            self.weights.grad[token_id] += grad_out.flatten()[i]

    def parameters(self):
        return [self.weights]



class PositionalEmbeddings:

    def __init__(self,seq_len,num_dims):
        self.weights  = torch.randn(seq_len,num_dims,requires_grad=True)

    def forward(self,positions):
        self.positions = positions
        return self.weights[positions]

    def backward(self,grad_out):
        
        self.grad_weights = torch.zeros_like(self.weights)
        
        for i,pos in enumerate(self.positions.flatten()):
            self.grad_weights[pos]+=grad_out.flatten()[i]
        
    def parameters(self):
        return [self.weights]



B = 4  #batch size (how many it needs to process at a time in parallel)
seq_len=24  #this is the total how many words can enter once 
vocab_size=1000
num_dims=64

# create the objects for those respective classes 

toe = TokenEmbeddings(vocab_size,num_dims)
poe = PositionalEmbeddings(seq_len,num_dims)

# randint give the integer values,ranging from the 0 to vocab_size-1 and then it expects the shape of the embeddings too

tok_embed = toe.forward(torch.randint(0,vocab_size,(B,seq_len)))


# so for the positions,we use the arange as it is same as the for loop of how it assigns the i values 
# we get the 1d tensor of [0,1,....23]
# use the unsqueeze to add the dimensions so it has a shape of the (1,24)
# so expand works like the repetition of the same 1 as B times and -1 says to keep that dim 


positions = torch.arange(seq_len).unsqueeze(0).expand(B,-1) 


pos_embed = poe.forward(positions)

x=  tok_embed+pos_embed

x.shape


